In [1]:
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm

# torch-int: CUTLASS INT8 GEMM kernels
# torch-int's repo is from 2023 and needs two patches to build on 2026 Colab:
#   (1) the cutlass submodule URL is SSH (git@github.com:...) which fails without an SSH key
#       → rewrite to HTTPS before submodule update
#   (2) setup.py hard-codes -std=c++14 but PyTorch 2.x requires C++17
#       → sed setup.py to c++17
%cd /content
# clone WITHOUT --recursive so the SSH submodule doesn't fail the whole clone
!git clone https://github.com/Guangxuan-Xiao/torch-int.git
%cd /content/torch-int

# Patch 1: rewrite SSH submodule URL → HTTPS, then init submodules
!git config --global url."https://github.com/".insteadOf "git@github.com:"
!sed -i 's|git@github.com:|https://github.com/|g' .gitmodules
!git submodule sync
!git submodule update --init --recursive

# Patch 2: C++14 → C++17 (setup.py and any kernel build flags)
!sed -i 's/c++14/c++17/g' setup.py
!grep -rl 'c++14' torch_int/ submodules/cutlass/CMakeLists.txt 2>/dev/null | xargs -r sed -i 's/c++14/c++17/g' || true

# Build CUTLASS test wrappers (build_cutlass.sh runs cmake on submodules/cutlass)
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5;8.0"  # T4=7.5, A100=8.0
!bash environment.sh || true   # pip deps; some lines may fail on Colab — ok
!bash build_cutlass.sh

# Build the python extension
!python setup.py install 2>&1 | tail -50

# Verify the CUDA extension actually compiled
!python -c "import torch_int._CUDA; print('torch_int._CUDA OK')"

%cd /content/llm-quantization-thesis

import sys
sys.path.insert(0, "/content/llm-quantization-thesis/smoothquant_repo")
sys.path.insert(0, "/content/llm-quantization-thesis")  # for experiments.task02_*

from google.colab import drive
drive.mount('/content/drive')

# Pull our percentile scales from Drive (Task 02 winner = p=0.999)
!mkdir -p act_percentiles/opt-1.3b
!cp /content/drive/MyDrive/thesis_results/act_percentiles/opt-1.3b/p0.999.pt act_percentiles/opt-1.3b/

# Pile val set for static calibration (Path B)
!mkdir -p dataset
!cp /content/drive/MyDrive/thesis_results/datasets/val.jsonl.zst dataset/ 2>/dev/null || \
   (echo "Pile val.jsonl.zst not on Drive. Downloading..." && \
    wget -q https://mystic.the-eye.eu/public/AI/pile/val.jsonl.zst -O dataset/val.jsonl.zst && \
    cp dataset/val.jsonl.zst /content/drive/MyDrive/thesis_results/datasets/)

!nvidia-smi
!python -c "import torch_int; print('torch-int OK')"
!python -c "from smoothquant.opt import Int8OPTForCausalLM; print('Int8OPTForCausalLM OK')"

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 192 (delta 79), reused 165 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 4.89 MiB | 21.13 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 26.07 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
/content
Cloning into 'torch-int'...
remote: Enumerating objects: 1102, done.
remote: Counting object

In [2]:
%cd /content/torch-int
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5;8.0"
os.environ["CPATH"] = (
    "/content/torch-int/submodules/cutlass/include:"
    "/content/torch-int/submodules/cutlass/tools/util/include:"
    + os.environ.get("CPATH", "")
)
!CPATH=$CPATH python setup.py install 2>&1 | tail -60
!python -c "import torch_int._CUDA; print('torch_int._CUDA OK')"

/content/torch-int
running build_ext
W0509 21:55:11.712000 19132 torch/utils/cpp_extension.py:535] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.8
building 'torch_int._CUDA' extension
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -Itorch_int/kernels/include -I/usr/local/lib/python3.12/dist-packages/torch/include -I/usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -I/usr/local/cuda/include -I/usr/include/python3.12 -c torch_int/kernels/bindings.cpp -o build/temp.linux-x86_64-cpython-312/torch_int/kernels/bindings.o -std=c++17 -O3 -DTORCH_API_INCLUDE_EXTENSION_H -DTORCH_EXTENSION_NAME=_CUDA
/usr/local/cuda/bin/nvcc -Itorch_int/kernels/include -I/usr/local/lib/python3.12/dist-packages/torch/include -I/usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -I/usr/local/cuda/include -I/usr/include/py

In [3]:
%cd /content
!python -c "import torch_int._CUDA; print('torch_int._CUDA OK')"
!python -c "from smoothquant.opt import Int8OPTForCausalLM; print('Int8OPTForCausalLM OK')"

/content
torch_int._CUDA OK
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/llm-quantization-thesis/smoothquant_repo/smoothquant/opt.py", line 319, in <module>
    class Int8OPTDecoder(OPTPreTrainedModel):
  File "/content/llm-quantization-thesis/smoothquant_repo/smoothquant/opt.py", line 375, in Int8OPTDecoder
    _prepare_decoder_attention_mask = OPTDecoder._prepare_decoder_attention_mask
                                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: type object 'OPTDecoder' has no attribute '_prepare_decoder_attention_mask'


In [4]:
!pip install -q "transformers==4.34.1"
# Verify and re-import
!python -c "import transformers; print('transformers', transformers.__version__)"
!python -c "from smoothquant.opt import Int8OPTForCausalLM; print('Int8OPTForCausalLM OK')"

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/torch_int-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 158.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 139.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 33.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.17.3 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.17.3 which is

In [5]:
%cd /content/llm-quantization-thesis
!mkdir -p dataset /content/drive/MyDrive/thesis_results/datasets
import os, json, zstandard as zstd
from datasets import load_dataset

if os.path.exists("/content/drive/MyDrive/thesis_results/datasets/val.jsonl.zst"):
    !cp /content/drive/MyDrive/thesis_results/datasets/val.jsonl.zst dataset/
    print("Loaded cached Pile sample from Drive.")
else:
    print("Building Pile sample from NeelNanda/pile-10k on HuggingFace...")
    ds = load_dataset("NeelNanda/pile-10k", split="train")
    cctx = zstd.ZstdCompressor(level=3)
    with open("dataset/val.jsonl.zst", "wb") as fh:
        with cctx.stream_writer(fh) as zf:
            for ex in ds:
                zf.write((json.dumps({"text": ex["text"]}) + "\n").encode("utf-8"))
    !cp dataset/val.jsonl.zst /content/drive/MyDrive/thesis_results/datasets/
    print("Saved val.jsonl.zst to Drive for reuse.")
!ls -lh dataset/val.jsonl.zst

/content/llm-quantization-thesis


ImportError: cannot import name 'insecure_hashlib' from 'huggingface_hub.utils' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/__init__.py)

In [7]:
!pip install -q "transformers==4.34.1" "datasets==2.14.7" "huggingface-hub==0.17.3" "tokenizers==0.14.1"
!python -c "import transformers, datasets, huggingface_hub, tokenizers; \
print('transformers', transformers.__version__, '| datasets', datasets.__version__, \
'| hub', huggingface_hub.__version__, '| tokenizers', tokenizers.__version__)"
!python -c "from smoothquant.opt import Int8OPTForCausalLM; print('Int8OPTForCausalLM OK')"

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/torch_int-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.4/520.4 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 17.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.17.3 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.17.3 which

In [2]:
%cd /content/llm-quantization-thesis
!mkdir -p dataset /content/drive/MyDrive/thesis_results/datasets
import os, json, zstandard as zstd
from datasets import load_dataset

if os.path.exists("/content/drive/MyDrive/thesis_results/datasets/val.jsonl.zst"):
    !cp /content/drive/MyDrive/thesis_results/datasets/val.jsonl.zst dataset/
    print("Loaded cached Pile sample from Drive.")
else:
    print("Building Pile sample from NeelNanda/pile-10k on HuggingFace...")
    ds = load_dataset("NeelNanda/pile-10k", split="train")
    cctx = zstd.ZstdCompressor(level=3)
    with open("dataset/val.jsonl.zst", "wb") as fh:
        with cctx.stream_writer(fh) as zf:
            for ex in ds:
                zf.write((json.dumps({"text": ex["text"]}) + "\n").encode("utf-8"))
    !cp dataset/val.jsonl.zst /content/drive/MyDrive/thesis_results/datasets/
    print("Saved val.jsonl.zst to Drive for reuse.")
!ls -lh dataset/val.jsonl.zst

/content/llm-quantization-thesis
Building Pile sample from NeelNanda/pile-10k on HuggingFace...


Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Saved val.jsonl.zst to Drive for reuse.
-rw-r--r-- 1 root root 21M May  9 22:18 dataset/val.jsonl.zst


In [1]:
import sys, os
sys.path.insert(0, "/content/llm-quantization-thesis/smoothquant_repo")
sys.path.insert(0, "/content/llm-quantization-thesis")

from google.colab import drive
drive.mount('/content/drive')

# Sanity checks — should all print clean versions and "OK"
import transformers, datasets, huggingface_hub, tokenizers
print('transformers', transformers.__version__,
      '| datasets', datasets.__version__,
      '| hub', huggingface_hub.__version__,
      '| tokenizers', tokenizers.__version__)

import torch_int._CUDA
print('torch_int._CUDA OK')
from smoothquant.opt import Int8OPTForCausalLM
print('Int8OPTForCausalLM OK')

%cd /content/llm-quantization-thesis

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
transformers 4.34.1 | datasets 2.14.7 | hub 0.17.3 | tokenizers 0.14.1
torch_int._CUDA OK
Int8OPTForCausalLM OK
/content/llm-quantization-thesis


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [3]:
!mkdir -p act_percentiles/opt-1.3b
!cp /content/drive/MyDrive/thesis_results/act_percentiles/opt-1.3b/p0.999.pt act_percentiles/opt-1.3b/
!ls -lh act_percentiles/opt-1.3b/

total 204K
-rw------- 1 root root 204K May  9 22:18 p0.999.pt


In [19]:
MODEL = "facebook/opt-1.3b"
RUNNER = "experiments/task04_real_int8_inference/run_real_int8_eval.py"
EXPORTER = "experiments/task04_real_int8_inference/export_our_int8.py"

# Task 02 OPT-1.3B winner
P_PCT     = 0.999
ALPHA_PCT = 0.5
PCT_SCALES = f"act_percentiles/opt-1.3b/p{P_PCT}.pt"

# HF prequantized paper model
HF_INT8 = "mit-han-lab/opt-1.3b-smoothquant"

# Where our local INT8 export will be saved
LOCAL_INT8 = "int8_models/opt-1.3b-ours"

OUT_DIR = "results/task04"
!mkdir -p {OUT_DIR}

print(f"Ours: smooth_lm_pct  p={P_PCT}  alpha={ALPHA_PCT}  scales={PCT_SCALES}")

Ours: smooth_lm_pct  p=0.999  alpha=0.5  scales=act_percentiles/opt-1.3b/p0.999.pt


In [5]:
!python {RUNNER} \
    --mode fp16 \
    --model_path {MODEL} \
    --tokenizer_path {MODEL} \
    --config_label FP16 \
    --save_json {OUT_DIR}/opt-1.3b_realint8_FP16.json

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
[FP16] model size: 2509.61 MB  (load 13.3s)
Extracting data files: 100% 3/3 [00:00<00:00, 2290.30it/s]
Generating test split: 100% 4358/4358 [00:00<00:00, 470326.70 examples/s]
Generating train split: 100% 36718/36718 [00:00<00:00, 748945.71 examples/s]
Generating validation split: 100% 3760/3760 [00:00<00:00, 704862.03 examples/s]
[FP16] wikitext-2 PPL @ 2048: 14.6240

Extracting data files: 100% 3/3 [00:00<00:00, 1909.68it/s]
Generating train split: 100% 2662/2662 [00:05<00:00, 4

In [9]:
!python {RUNNER} \
    --mode fp16 \
    --model_path {MODEL} \
    --tokenizer_path {MODEL} \
    --config_label FP16 \
    --skip_lambada \
    --save_json {OUT_DIR}/opt-1.3b_realint8_FP16.json

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
[FP16] model size: 2509.61 MB  (load 6.2s)
[FP16] wikitext-2 PPL @ 2048: 14.6240
[FP16] peak VRAM alloc:    4557.4 MB
[FP16] peak VRAM reserved: 4964.0 MB
[FP16] activation peak:    2045.6 MB (peak_alloc - model_baseline)
saved -> results/task04/opt-1.3b_realint8_FP16.json


In [10]:
!python {RUNNER} \
    --mode int8_hf \
    --model_path {HF_INT8} \
    --tokenizer_path {MODEL} \
    --config_label INT8-paper \
    --skip_lambada \
    --save_json {OUT_DIR}/opt-1.3b_realint8_INT8-paper.json

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
[INT8-paper] model size: 1357.84 MB  (load 20.6s)
[INT8-paper] wikitext-2 PPL @ 2048: 18.0245
[INT8-paper] peak VRAM alloc:    3140.8 MB
[INT8-paper] peak VRAM reserved: 3362.0 MB
[INT8-paper] activation peak:    1780.6 MB (peak_alloc - model_baseline)
saved -> results/task04/opt-1.3b_realint8_INT8-paper.json


In [8]:
%cd /content/llm-quantization-thesis
!git pull

/content/llm-quantization-thesis
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.67 KiB | 1.33 MiB/s, done.
From https://github.com/Anand-786/llm-quantization-thesis
   813ebdc..2913c04  main       -> origin/main
Updating 813ebdc..2913c04
Fast-forward
 .../opt_1_3b/real_int8_cells.md                    | 80 ++++++++++++++++++----
 .../run_real_int8_eval.py                          | 42 ++++++++++--
 2 files changed, 104 insertions(+), 18 deletions(-)


In [20]:
!python {EXPORTER} \
    --model_name {MODEL} \
    --pct_scales {PCT_SCALES} \
    --alpha {ALPHA_PCT} \
    --p_w {P_PCT} \
    --dataset_path dataset/val.jsonl.zst \
    --num_samples 512 \
    --seq_len 512 \
    --output_path {LOCAL_INT8}

!ls -lh {LOCAL_INT8}

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
loading facebook/opt-1.3b (fp16)
loading percentile scales from act_percentiles/opt-1.3b/p0.999.pt
applying percentile smoothing  alpha=0.5  p_w=0.999
static calibration on dataset/val.jsonl.zst (512 samples x 512 tokens)
Mean input scale: 4.55: 100% 512/512 [00:33<00:00, 15.16it/s]
converting to Int8OPTForCausalLM
saved INT8 model to int8_models/opt-1.3b-ours
total 1.4G
-rw-r--r-- 1 root root  753 May  9 23:28 config.json
-rw-r--r-- 1 root root  132 May  9 23:28 generation_config.

In [12]:
%%writefile /content/llm-quantization-thesis/experiments/task04_real_int8_inference/export_our_int8.py
"""Export an Int8OPTForCausalLM checkpoint smoothed with our percentile method."""
import argparse
import os
import sys
from pathlib import Path

# Make the repo root importable so `from experiments.task02_...` works when
# this script is launched as `python experiments/task04_.../export_our_int8.py`.
_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

import torch
from transformers import AutoTokenizer
from transformers.models.opt.modeling_opt import OPTForCausalLM

from smoothquant.opt import Int8OPTForCausalLM
from smoothquant.calibration import get_static_decoder_layer_scales
from experiments.task02_percentile_smoothing.percentile_smooth import smooth_lm_pct


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_name", required=True)
    ap.add_argument("--pct_scales", required=True)
    ap.add_argument("--alpha", type=float, required=True)
    ap.add_argument("--p_w", type=float, required=True)
    ap.add_argument("--dataset_path", required=True)
    ap.add_argument("--num_samples", type=int, default=512)
    ap.add_argument("--seq_len", type=int, default=512)
    ap.add_argument("--output_path", required=True)
    args = ap.parse_args()

    print(f"loading {args.model_name} (fp16)")
    model = OPTForCausalLM.from_pretrained(
        args.model_name, device_map="auto", torch_dtype=torch.float16
    )
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)

    print(f"loading percentile scales from {args.pct_scales}")
    pct_scales = torch.load(args.pct_scales)

    print(f"applying percentile smoothing  alpha={args.alpha}  p_w={args.p_w}")
    smooth_lm_pct(model, pct_scales, alpha=args.alpha, p_w=args.p_w)

    if not os.path.exists(args.dataset_path):
        raise FileNotFoundError(args.dataset_path)

    print(f"static calibration on {args.dataset_path} "
          f"({args.num_samples} samples x {args.seq_len} tokens)")
    decoder_layer_scales, _raw = get_static_decoder_layer_scales(
        model, tokenizer, args.dataset_path,
        num_samples=args.num_samples, seq_len=args.seq_len,
    )

    print("converting to Int8OPTForCausalLM")
    int8_model = Int8OPTForCausalLM.from_float(model, decoder_layer_scales)

    out = Path(args.output_path)
    out.mkdir(parents=True, exist_ok=True)
    int8_model.save_pretrained(out)
    tokenizer.save_pretrained(out)
    print(f"saved INT8 model to {out}")


if __name__ == "__main__":
    main()


Overwriting /content/llm-quantization-thesis/experiments/task04_real_int8_inference/export_our_int8.py


In [13]:
!python -c "from experiments.task02_percentile_smoothing.percentile_smooth import smooth_lm_pct; import inspect; print(inspect.signature(smooth_lm_pct))"


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
(model, scales, alpha=0.5, p_w=1.0)


In [21]:
!python {RUNNER} \
    --mode int8_local \
    --model_path {LOCAL_INT8} \
    --tokenizer_path {MODEL} \
    --config_label INT8-ours \
    --skip_lambada \
    --save_json {OUT_DIR}/opt-1.3b_realint8_INT8-ours.json

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
  shape-repaired 144 entries; 0 unfixable
[INT8-ours] model size: 1357.47 MB  (load 20.5s)
[INT8-ours] wikitext-2 PPL @ 2048: 17.9588
[INT8-ours] peak VRAM alloc:    3140.4 MB
[INT8-ours] peak VRAM reserved: 3348.0 MB
[INT8-ours] activation peak:    1780.6 MB
saved -> results/task04/opt-1.3b_realint8_INT8-ours.json


In [16]:
%%writefile /content/llm-quantization-thesis/experiments/task04_real_int8_inference/run_real_int8_eval.py
"""Real-INT8 evaluation runner for Task 04."""
import argparse, gc, json, time, os
import torch
from torch.nn.functional import pad


def model_size_mb(model):
    p = sum(x.nelement() * x.element_size() for x in model.parameters())
    b = sum(x.nelement() * x.element_size() for x in model.buffers())
    return (p + b) / (1024 ** 2)


@torch.no_grad()
def eval_wikitext_ppl(model, tokenizer, seq_len=2048, device="cuda"):
    from datasets import load_dataset
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(ds["text"])
    enc = tokenizer(text, return_tensors="pt").input_ids.to(device)
    n_tokens = enc.shape[1]
    n_windows = n_tokens // seq_len
    nlls = []
    model.eval()

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats(device)
    baseline_alloc = torch.cuda.memory_allocated(device)

    for i in range(n_windows):
        ids = enc[:, i * seq_len : (i + 1) * seq_len]
        out = model(ids, labels=ids)
        nlls.append(out.loss.float() * seq_len)

    torch.cuda.synchronize()
    peak_alloc = torch.cuda.max_memory_allocated(device)
    peak_reserved = torch.cuda.max_memory_reserved(device)

    ppl = torch.exp(torch.stack(nlls).sum() / (n_windows * seq_len))
    return {
        "ppl": ppl.item(),
        "peak_vram_alloc_mb": peak_alloc / (1024 ** 2),
        "peak_vram_reserved_mb": peak_reserved / (1024 ** 2),
        "baseline_alloc_mb": baseline_alloc / (1024 ** 2),
        "activation_peak_mb": (peak_alloc - baseline_alloc) / (1024 ** 2),
    }


@torch.no_grad()
def eval_lambada(model, tokenizer, n_samples=1000, pad_to=512, device="cuda"):
    from datasets import load_dataset
    ds = load_dataset("lambada", split=f"validation[:{n_samples}]")
    def tok(ex): return tokenizer(ex["text"])
    ds = ds.map(tok, batched=True)
    ds.set_format(type="torch", columns=["input_ids"])
    total = hit = 0
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    latency_ms = 0.0
    model.eval()
    for batch in ds:
        ids = batch["input_ids"].to(device).unsqueeze(0)
        label = ids[:, -1]
        pad_len = pad_to - ids.shape[1]
        if pad_len < 0:
            ids = ids[:, :pad_to]; pad_len = 0
        ids = pad(ids, (0, pad_len), value=1)
        torch.cuda.synchronize(); start.record()
        out = model(ids)
        end.record(); torch.cuda.synchronize()
        latency_ms += start.elapsed_time(end)
        last_logits = out.logits[:, -2 - pad_len, :]
        pred = last_logits.argmax(dim=-1)
        total += label.size(0)
        hit += (pred == label).sum().item()
    return hit / total, latency_ms / len(ds)


def _load_int8_local_with_shape_repair(model_path):
    """torch-int's from_float -> save_pretrained -> from_pretrained loses some
    bias 2D shapes. Build the model from config, then load state dict with
    shape repair and strict=False."""
    from smoothquant.opt import Int8OPTForCausalLM
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(model_path)
    model = Int8OPTForCausalLM(config)
    sd = torch.load(os.path.join(model_path, "pytorch_model.bin"), map_location="cpu")
    msd = model.state_dict()
    repaired = mismatched = 0
    for k in list(sd.keys()):
        if k in msd and msd[k].shape != sd[k].shape:
            try:
                sd[k] = sd[k].reshape(msd[k].shape)
                repaired += 1
            except Exception:
                mismatched += 1
    print(f"  shape-repaired {repaired} entries; {mismatched} unfixable")
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing: print(f"  missing keys: {len(missing)} (e.g. {missing[:3]})")
    if unexpected: print(f"  unexpected keys: {len(unexpected)} (e.g. {unexpected[:3]})")
    return model.half().cuda()


def load_model(args):
    if args.mode == "fp16":
        from transformers.models.opt.modeling_opt import OPTForCausalLM
        return OPTForCausalLM.from_pretrained(args.model_path, torch_dtype=torch.float16, device_map="auto")
    elif args.mode == "int8_hf":
        from smoothquant.opt import Int8OPTForCausalLM
        return Int8OPTForCausalLM.from_pretrained(args.model_path, torch_dtype=torch.float16, device_map="auto")
    elif args.mode == "int8_local":
        return _load_int8_local_with_shape_repair(args.model_path)
    raise ValueError(args.mode)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", required=True, choices=["fp16", "int8_hf", "int8_local"])
    ap.add_argument("--model_path", required=True)
    ap.add_argument("--tokenizer_path", required=True)
    ap.add_argument("--config_label", required=True)
    ap.add_argument("--save_json", required=True)
    ap.add_argument("--seq_len_ppl", type=int, default=2048)
    ap.add_argument("--lambada_samples", type=int, default=1000)
    ap.add_argument("--skip_ppl", action="store_true")
    ap.add_argument("--skip_lambada", action="store_true")
    args = ap.parse_args()

    from transformers import GPT2Tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained(args.tokenizer_path)

    t0 = time.time()
    model = load_model(args)
    load_s = time.time() - t0

    size_mb = model_size_mb(model)
    print(f"[{args.config_label}] model size: {size_mb:.2f} MB  (load {load_s:.1f}s)")

    result = {"config_label": args.config_label, "mode": args.mode,
              "model_path": args.model_path, "size_mb": size_mb,
              "seq_len_ppl": args.seq_len_ppl}

    if not args.skip_ppl:
        m = eval_wikitext_ppl(model, tokenizer, seq_len=args.seq_len_ppl)
        print(f"[{args.config_label}] wikitext-2 PPL @ {args.seq_len_ppl}: {m['ppl']:.4f}")
        print(f"[{args.config_label}] peak VRAM alloc:    {m['peak_vram_alloc_mb']:.1f} MB")
        print(f"[{args.config_label}] peak VRAM reserved: {m['peak_vram_reserved_mb']:.1f} MB")
        print(f"[{args.config_label}] activation peak:    {m['activation_peak_mb']:.1f} MB")
        result["wikitext2_ppl"] = m["ppl"]
        result["peak_vram_alloc_mb"] = m["peak_vram_alloc_mb"]
        result["peak_vram_reserved_mb"] = m["peak_vram_reserved_mb"]
        result["activation_peak_mb"] = m["activation_peak_mb"]
        result["baseline_alloc_mb"] = m["baseline_alloc_mb"]

    if not args.skip_lambada:
        acc, lat = eval_lambada(model, tokenizer, n_samples=args.lambada_samples)
        print(f"[{args.config_label}] LAMBADA acc: {acc:.4f}  latency: {lat:.3f} ms/sample")
        result["lambada_last_token_acc"] = acc
        result["lambada_latency_ms_per_sample"] = lat

    with open(args.save_json, "w") as f:
        json.dump(result, f, indent=2)
    print(f"saved -> {args.save_json}")
    del model; gc.collect(); torch.cuda.empty_cache()


if __name__ == "__main__":
    main()


Overwriting /content/llm-quantization-thesis/experiments/task04_real_int8_inference/run_real_int8_eval.py


In [23]:
!mkdir -p /content/drive/MyDrive/thesis_results/task04
!cp {OUT_DIR}/opt-1.3b_realint8_*.json /content/drive/MyDrive/thesis_results/task04/

# Optionally back up the exported INT8 weights too — large file, ~1.3GB
# !cp -r {LOCAL_INT8} /content/drive/MyDrive/thesis_results/task04/

import json, glob

ORDER = ["FP16", "INT8-paper", "INT8-ours"]
COLS  = ["size_mb", "peak_vram_alloc_mb", "activation_peak_mb", "wikitext2_ppl"]

rows_by_label = {}
for f in sorted(glob.glob(f"{OUT_DIR}/opt-1.3b_realint8_*.json")):
    r = json.load(open(f))
    rows_by_label[r["config_label"]] = r

header = ["config"] + COLS
print("  ".join(f"{h:>30}" for h in header))
print("-" * (32 * len(header)))
for label in ORDER:
    r = rows_by_label.get(label)
    if r is None:
        continue
    cells = [f"{label:>30}"]
    for c in COLS:
        v = r.get(c)
        cells.append(f"{v:>30.4f}" if isinstance(v, (int, float)) else f"{'-':>30}")
    print("  ".join(cells))

# Headline numbers for the thesis
fp16 = rows_by_label.get("FP16", {})
ours = rows_by_label.get("INT8-ours", {})
paper = rows_by_label.get("INT8-paper", {})
if fp16 and ours:
    print()
    print(f"--- Headline ratios (lower = better, ~50% is the SmoothQuant target) ---")
    print(f"  Static model size:  INT8-ours / FP16 = "
          f"{ours['size_mb']/fp16['size_mb']*100:.1f}%")
    if "peak_vram_alloc_mb" in fp16 and "peak_vram_alloc_mb" in ours:
        print(f"  Peak inference VRAM: INT8-ours / FP16 = "
              f"{ours['peak_vram_alloc_mb']/fp16['peak_vram_alloc_mb']*100:.1f}%  "
              f"({ours['peak_vram_alloc_mb']:.0f} MB vs {fp16['peak_vram_alloc_mb']:.0f} MB)")
    if "activation_peak_mb" in fp16 and "activation_peak_mb" in ours:
        print(f"  Activation peak:     INT8-ours / FP16 = "
              f"{ours['activation_peak_mb']/fp16['activation_peak_mb']*100:.1f}%  "
              f"({ours['activation_peak_mb']:.0f} MB vs {fp16['activation_peak_mb']:.0f} MB)")
    if paper:
        print()
        print(f"--- PPL: ours vs paper (lower = better, FP16={fp16.get('wikitext2_ppl', 0):.2f}) ---")
        print(f"  Paper O3+max:    {paper.get('wikitext2_ppl', 0):.4f}")
        print(f"  Ours O3+pct:     {ours.get('wikitext2_ppl', 0):.4f}  "
              f"(delta vs paper: {ours.get('wikitext2_ppl', 0) - paper.get('wikitext2_ppl', 0):+.4f})")

                        config                         size_mb              peak_vram_alloc_mb              activation_peak_mb                   wikitext2_ppl
----------------------------------------------------------------------------------------------------------------------------------------------------------------
                          FP16                       2509.6094                       4557.3804                       2045.5762                         14.6240
                    INT8-paper                       1357.8443                       3140.7554                       1780.5762                         18.0245
                     INT8-ours                       1357.4693                       3140.3804                       1780.5762                         17.9588

--- Headline ratios (lower = better, ~50% is the SmoothQuant target) ---
  Static model size:  INT8-ours / FP16 = 54.1%
  Peak inference VRAM: INT8-ours / FP16 = 68.9%  (3140 MB vs 4557 MB)
  Activation

In [24]:
# Backup all 4 JSONs to Drive (you may already have done this)
!mkdir -p /content/drive/MyDrive/thesis_results/task04
!cp results/task04/opt-1.3b_realint8_*.json /content/drive/MyDrive/thesis_results/task04/
!ls /content/drive/MyDrive/thesis_results/task04/

opt-1.3b_realint8_FP16.json	  opt-1.3b_realint8_INT8-paper.json
opt-1.3b_realint8_INT8-ours.json
